# Cross-Validation with a Second, Independent NLI Model

Addresses Examiner 1's (Aditya Kurniawan, D3448) notulen sidang butir #1 — the circularity concern: *"NLI digunakan untuk memilih kandidat ringkasan yang memiliki skor entailment tinggi. Setelah kandidat dipilih, keberhasilannya dinilai menggunakan skor dari NLI yang sama... bukti apa yang menunjukkan bahwa kenaikan skor tersebut juga diikuti berkurangnya kesalahan fakta?"* The same concern was raised as Reviewer 1's Major Comment #1 in the CompGineer journal review.

The selection/evaluation model throughout this project is `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`. This notebook re-scores both the baseline and BART+NLI summaries using a **second, architecturally independent** NLI model — `joeddav/xlm-roberta-large-xnli` (XLM-RoBERTa-large, a different model family from DeBERTa, fine-tuned on XNLI, covers Indonesian) — and checks whether the entailment improvement still holds when judged by a model that had no role in selecting the candidates.

This notebook is self-contained: it recomputes entailment for **both** conditions with **both** models from the raw `document`/`generated_summary` text already in `predictions_baseline.jsonl` / `predictions_nli.jsonl`, so it does not depend on any other notebook's session state (the stored `entailment_score` field in `predictions_baseline.jsonl` is unusable directly — it is always `0.0` due to a known pipeline bug documented in `05_significance_test.ipynb` and `07_rouge_entailment_correlation.ipynb`).

## 1. Setup

In [ ]:
!pip install -q transformers

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

## 2. Load Predictions

In [ ]:
import json
import glob
from pathlib import Path

# Flat dataset attached via Add Input (same "prediction-21092026" dataset used by
# 06_claim_level_hallucination.ipynb / 07_rouge_entailment_correlation.ipynb).
# Falls back to a recursive glob if Kaggle mounted it under a different slug/path.
PREDICTIONS_BASELINE_FILE = "/kaggle/input/datasets/madedwikibudilaksana/prediction-21092026/predictions_baseline.jsonl"
PREDICTIONS_NLI_FILE = "/kaggle/input/datasets/madedwikibudilaksana/prediction-21092026/predictions_nli.jsonl"

if not Path(PREDICTIONS_BASELINE_FILE).exists():
    hits = glob.glob("/kaggle/input/**/predictions_baseline.jsonl", recursive=True)
    if hits:
        PREDICTIONS_BASELINE_FILE = hits[0]
if not Path(PREDICTIONS_NLI_FILE).exists():
    hits = glob.glob("/kaggle/input/**/predictions_nli.jsonl", recursive=True)
    if hits:
        PREDICTIONS_NLI_FILE = hits[0]

if not Path(PREDICTIONS_BASELINE_FILE).exists() or not Path(PREDICTIONS_NLI_FILE).exists():
    print("Not found. Contents of /kaggle/input:")
    for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        print(" ", p)
    raise FileNotFoundError("predictions_baseline.jsonl / predictions_nli.jsonl not found under /kaggle/input.")

print("Baseline file:", PREDICTIONS_BASELINE_FILE)
print("NLI file:", PREDICTIONS_NLI_FILE)

def load_by_id(path):
    rows = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                row = json.loads(line)
                rows[row["id"]] = row
    return rows

baseline_rows = load_by_id(PREDICTIONS_BASELINE_FILE)
nli_rows = load_by_id(PREDICTIONS_NLI_FILE)
shared_ids = sorted(set(baseline_rows.keys()) & set(nli_rows.keys()))
print(f"Baseline: {len(baseline_rows)} | NLI: {len(nli_rows)} | Shared (paired): {len(shared_ids)}")

## 3. Load Both NLI Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_nli_model(model_name, batch_size):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(DEVICE).eval()
    id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
    ent_idx = next(i for i, l in id2label.items() if "entail" in l)
    con_idx = next(i for i, l in id2label.items() if "contrad" in l)
    print(f"Loaded {model_name} on {DEVICE} (entailment idx={ent_idx}, contradiction idx={con_idx}, batch size {batch_size})")
    return tokenizer, model, ent_idx, con_idx

# Model 1: the SAME model used throughout the project for candidate selection/evaluation.
# Recomputed here (not read from the buggy stored field) so both models are scored identically.
MODEL1_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
model1_tokenizer, model1_model, model1_ent_idx, model1_con_idx = load_nli_model(MODEL1_NAME, batch_size=32)

# Model 2: an independent model that played NO role in candidate selection.
# Different architecture family (XLM-RoBERTa vs. DeBERTa), fine-tuned on XNLI (covers Indonesian).
MODEL2_NAME = "joeddav/xlm-roberta-large-xnli"
model2_tokenizer, model2_model, model2_ent_idx, model2_con_idx = load_nli_model(MODEL2_NAME, batch_size=16)

@torch.inference_mode()
def score_entailment_batch(tokenizer, model, ent_idx, premises, hypotheses):
    enc = tokenizer(premises, hypotheses, truncation=True, max_length=512, padding=True, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    probs = torch.softmax(model(**enc).logits, dim=-1)
    return probs[:, ent_idx].tolist()

## 4. Re-score Both Conditions with Both Models

In [ ]:
import statistics
import time

def rescore_all(rows, ids, tokenizer, model, ent_idx, batch_size, label):
    scores = {}
    t0 = time.time()
    for i in range(0, len(ids), batch_size):
        batch_ids = ids[i:i + batch_size]
        premises = [rows[sid]["document"] for sid in batch_ids]
        hypotheses = [rows[sid]["generated_summary"] for sid in batch_ids]
        ents = score_entailment_batch(tokenizer, model, ent_idx, premises, hypotheses)
        for sid, ent in zip(batch_ids, ents):
            scores[sid] = ent
        done = i + len(batch_ids)
        if done % (batch_size * 20) < batch_size:
            elapsed = time.time() - t0
            rate = elapsed / done
            remaining = rate * (len(ids) - done)
            print(f"  [{label}] {done}/{len(ids)} ({rate:.3f}s/sample, ~{remaining/60:.1f} min remaining)", flush=True)
    return scores

print("=== Model 1 (mDeBERTa, original selection/evaluation model) ===")
model1_baseline_scores = rescore_all(baseline_rows, shared_ids, model1_tokenizer, model1_model, model1_ent_idx, 32, "model1-baseline")
model1_nli_scores = rescore_all(nli_rows, shared_ids, model1_tokenizer, model1_model, model1_ent_idx, 32, "model1-nli")

print("\n=== Model 2 (XLM-RoBERTa-large-XNLI, independent evaluator) ===")
model2_baseline_scores = rescore_all(baseline_rows, shared_ids, model2_tokenizer, model2_model, model2_ent_idx, 16, "model2-baseline")
model2_nli_scores = rescore_all(nli_rows, shared_ids, model2_tokenizer, model2_model, model2_ent_idx, 16, "model2-nli")

m1_baseline_mean = statistics.mean(model1_baseline_scores.values())
m1_nli_mean = statistics.mean(model1_nli_scores.values())
m2_baseline_mean = statistics.mean(model2_baseline_scores.values())
m2_nli_mean = statistics.mean(model2_nli_scores.values())

m1_rel_gain = (m1_nli_mean - m1_baseline_mean) / m1_baseline_mean * 100
m2_rel_gain = (m2_nli_mean - m2_baseline_mean) / m2_baseline_mean * 100

print("\n=== Summary ===")
print(f"Model 1 (mDeBERTa)          : baseline={m1_baseline_mean:.4f} -> NLI={m1_nli_mean:.4f}  ({m1_rel_gain:+.1f}%)")
print(f"Model 2 (XLM-R-large-XNLI)  : baseline={m2_baseline_mean:.4f} -> NLI={m2_nli_mean:.4f}  ({m2_rel_gain:+.1f}%)")
print("(sanity check: Model 1's baseline mean should be close to 0.3100, per Table IV / main_results.json;")
print(" Model 1's NLI mean should be close to 0.4664.)")

## 5. Significance Test (Model 2) and Cross-Model Agreement

In [ ]:
from scipy import stats as scipy_stats
import numpy as np

# Paired t-test on Model 2's scores (baseline vs NLI) -- does the improvement hold up
# under an evaluator that had no role in selecting the candidates?
m2_baseline_list = [model2_baseline_scores[sid] for sid in shared_ids]
m2_nli_list = [model2_nli_scores[sid] for sid in shared_ids]
t_stat, p_value = scipy_stats.ttest_rel(m2_baseline_list, m2_nli_list)
diffs = np.array(m2_baseline_list) - np.array(m2_nli_list)
cohens_d = float(diffs.mean() / diffs.std(ddof=1))
p_str = f"{p_value:.2e}" if p_value >= 1e-300 else "< 1e-300"
print(f"Model 2 paired t-test: t={t_stat:.3f}, p={p_str}, Cohen's d={cohens_d:.3f} "
      f"(positive d means baseline > NLI, i.e. same sign convention as the rest of the project)")

# Correlation between Model 1's and Model 2's scores, per condition -- do the two
# independent models broadly agree on which summaries are more/less entailed?
def report_corr(x, y, label):
    r, p = scipy_stats.pearsonr(x, y)
    rho, ps = scipy_stats.spearmanr(x, y)
    print(f"[{label}] Model1 vs Model2: Pearson r={r:.4f} (p={p:.2e}) | Spearman rho={rho:.4f} (p={ps:.2e})")
    return {"pearson_r": round(float(r), 4), "pearson_p": float(p), "spearman_rho": round(float(rho), 4), "spearman_p": float(ps)}

print("\n=== Cross-model agreement ===")
corr_baseline = report_corr([model1_baseline_scores[s] for s in shared_ids], m2_baseline_list, "baseline")
corr_nli = report_corr([model1_nli_scores[s] for s in shared_ids], m2_nli_list, "BART+NLI")

## 6. Save Results

In [ ]:
output_path = Path("./results/second_nli_evaluator_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

payload = {
    "model1_name": MODEL1_NAME,
    "model2_name": MODEL2_NAME,
    "n_paired_samples": len(shared_ids),
    "model1": {"baseline_mean_entailment": round(m1_baseline_mean, 4), "nli_mean_entailment": round(m1_nli_mean, 4), "relative_gain_pct": round(m1_rel_gain, 1)},
    "model2": {"baseline_mean_entailment": round(m2_baseline_mean, 4), "nli_mean_entailment": round(m2_nli_mean, 4), "relative_gain_pct": round(m2_rel_gain, 1)},
    "model2_paired_ttest": {"t_statistic": round(float(t_stat), 3), "p_value": float(p_value), "cohens_d": round(cohens_d, 3)},
    "cross_model_agreement": {"baseline": corr_baseline, "nli": corr_nli},
}
with output_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
print(f"Saved to {output_path}")

## Narasi untuk disalin ke tesis (draf, sesuaikan dengan angka aktual hasil run)

Contoh kalimat untuk subbab 4.4 (menanggapi kekhawatiran sirkularitas, lihat draf sebelumnya di `Draft_Revisi_Naskah_Tesis.md` bagian 6), mengisi Examiner 1 #1:

> Untuk menjawab kekhawatiran sirkularitas secara langsung, dilakukan validasi silang dengan menilai ulang seluruh ringkasan *baseline* dan BART + NLI menggunakan model NLI kedua yang tidak berperan sama sekali dalam proses seleksi kandidat, yaitu `joeddav/xlm-roberta-large-xnli` (arsitektur XLM-RoBERTa-large, berbeda keluarga model dari mDeBERTa-v3 yang dipakai untuk seleksi). Model independen ini menghasilkan skor *entailment* rata-rata **[ISI]** untuk *baseline* dan **[ISI]** untuk BART + NLI, sebuah peningkatan relatif sebesar **[ISI]%** — [**bandingkan dengan 50,5% pada model asli: apakah sepadan, lebih kecil, atau lebih besar? ISI SESUAI HASIL**]. Perbedaan ini signifikan secara statistik (uji-t berpasangan, t=**[ISI]**, p **[ISI]**, Cohen's d=**[ISI]**). Selain itu, skor kedua model menunjukkan korelasi **[ISI: kuat/sedang/lemah]** (Pearson r=**[ISI]** pada kondisi *baseline*, r=**[ISI]** pada BART+NLI), menunjukkan bahwa kedua model NLI yang independen secara umum sepakat mengenai ringkasan mana yang lebih/kurang didukung dokumen sumber. Temuan ini [**perkuat/lemahkan, ISI SESUAI HASIL**] klaim bahwa peningkatan konsistensi faktual bukan sekadar artefak dari optimasi terhadap satu model penilai saja.

**Penting:** isi bagian `[ISI]` di atas SESUAI ARAH HASIL AKTUAL. Ada tiga kemungkinan hasil, dan masing-masing perlu ditulis dengan jujur:
1. **Model 2 juga menunjukkan peningkatan besar (mendekati 50%)** — ini bukti kuat bahwa temuan bukan artefak sirkularitas, tulis dengan percaya diri.
2. **Model 2 menunjukkan peningkatan tapi lebih kecil** — tulis apa adanya: arah temuan konsisten, tapi besarannya tidak sebesar model asli, kemungkinan karena model asli sedikit "menguntungkan" dirinya sendiri dalam evaluasi (bias yang sudah diakui di subbab 4.4/4.6). Ini tetap jawaban yang valid dan jujur untuk Examiner 1 #1.
3. **Model 2 tidak menunjukkan peningkatan berarti** — ini temuan penting yang HARUS dilaporkan, bukan disembunyikan. Artinya kekhawatiran sirkularitas Examiner 1 terbukti berdasar, dan klaim di tesis perlu diperlemah lebih jauh (konsisten dengan reframing "NLI-estimated source-groundedness" yang sudah diadopsi di bagian 6/7 draf revisi sebelumnya).